In [1]:
import torch
import torchvision
import torchvision.transforms as transforms
from torch.utils.data import DataLoader

In [2]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

Using device: cuda


In [3]:
cinic_dir = "./cinic10"

In [4]:
cinic_mean = [0.47889522, 0.47227842, 0.43047404]
cinic_std = [0.24205776, 0.23828046, 0.25874835]

transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(mean=cinic_mean, std=cinic_std)
])

In [5]:
train_dataset = torchvision.datasets.ImageFolder(
    cinic_dir + "/train",
    transform=transform
)

valid_dataset = torchvision.datasets.ImageFolder(
    cinic_dir + "/valid",
    transform=transform
)

train_loader = DataLoader(train_dataset, batch_size=128, shuffle=True)
valid_loader = DataLoader(valid_dataset, batch_size=128, shuffle=False)

In [6]:
import torch
import torch.nn as nn

class CNN(nn.Module):

    def __init__(self):
        super().__init__()

        self.features = nn.Sequential(

            nn.Conv2d(3, 64, 3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.Conv2d(64, 64, 3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(64, 128, 3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(),
            nn.Conv2d(128, 128, 3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(128, 256, 3, padding=1),
            nn.BatchNorm2d(256),
            nn.ReLU(),
            nn.Conv2d(256, 256, 3, padding=1),
            nn.BatchNorm2d(256),
            nn.ReLU(),
            nn.MaxPool2d(2),


            nn.Conv2d(256, 512, 3, padding=1),
            nn.BatchNorm2d(512),
            nn.ReLU(),
            nn.MaxPool2d(2)
        )


        self.classifier = nn.Sequential(

            nn.Linear(512 * 2 * 2, 1024),
            nn.ReLU(),
            nn.Dropout(0.5),

            nn.Linear(1024, 512),
            nn.ReLU(),
            nn.Dropout(0.5),

            nn.Linear(512, 10)
        )

    def forward(self, x):
        x = self.features(x)
        x = torch.flatten(x, 1)
        return self.classifier(x)


model = CNN().to(device)

In [7]:
criterion = nn.CrossEntropyLoss()

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=0.001
)

scheduler = torch.optim.lr_scheduler.StepLR(
    optimizer,
    step_size=5,
    gamma=0.5
)

In [8]:
class EarlyStopping:

    def __init__(self, patience=5, path="best_model.pth"):

        self.patience = patience
        self.path = path

        self.best_loss = float("inf")
        self.counter = 0
        self.stop = False

    def __call__(self, val_loss, model):

        if val_loss < self.best_loss:

            self.best_loss = val_loss
            self.counter = 0

            torch.save(model.state_dict(), self.path)

        else:
            self.counter += 1

            if self.counter >= self.patience:
                self.stop = True

In [9]:
def validate(model, loader):

    model.eval()
    total_loss = 0

    with torch.no_grad():

        for images, labels in loader:

            images = images.to(device)
            labels = labels.to(device)

            outputs = model(images)
            loss = criterion(outputs, labels)

            total_loss += loss.item()

    return total_loss / len(loader)

In [10]:
early_stopping = EarlyStopping(patience=5)

epochs = 20

for epoch in range(epochs):

    model.train()
    train_loss = 0

    for images, labels in train_loader:

        images = images.to(device)
        labels = labels.to(device)

        outputs = model(images)
        loss = criterion(outputs, labels)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        train_loss += loss.item()

    train_loss /= len(train_loader)

    val_loss = validate(model, valid_loader)

    print(f"\nEpoch {epoch+1}")
    print(f"Train Loss: {train_loss:.4f}")
    print(f"Val Loss: {val_loss:.4f}")

    early_stopping(val_loss, model)

    if early_stopping.stop:
        break

    scheduler.step()


Epoch 1
Train Loss: 1.6039
Val Loss: 1.3322

Epoch 2
Train Loss: 1.2624
Val Loss: 1.1379

Epoch 3
Train Loss: 1.0987
Val Loss: 1.0310

Epoch 4
Train Loss: 0.9750
Val Loss: 1.0144

Epoch 5
Train Loss: 0.8742
Val Loss: 0.8923

Epoch 6
Train Loss: 0.7118
Val Loss: 0.7914

Epoch 7
Train Loss: 0.6286
Val Loss: 0.7834

Epoch 8
Train Loss: 0.5605
Val Loss: 0.7580

Epoch 9
Train Loss: 0.4945
Val Loss: 0.7954

Epoch 10
Train Loss: 0.4301
Val Loss: 0.7743

Epoch 11
Train Loss: 0.3008
Val Loss: 0.8377

Epoch 12
Train Loss: 0.2452
Val Loss: 0.9237

Epoch 13
Train Loss: 0.2044
Val Loss: 1.0134


In [11]:
best_model = CNN().to(device)
best_model.load_state_dict(torch.load("best_model.pth"))
best_model.to(device)
best_model.eval()

correct = 0
total = 0

with torch.no_grad():

    for images, labels in valid_loader:

        images = images.to(device)
        labels = labels.to(device)

        outputs = best_model(images)

        _, predicted = torch.max(outputs, 1)

        total += labels.size(0)
        correct += (predicted == labels).sum().item()

print("Valid Accuracy:", 100 * correct / total)

Valid Accuracy: 73.58444444444444
